In [11]:
# cell 1 (note the last line)
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf repo
!git clone -b feature/ablations https://github.com/ruwini01/Sinhala_English_Code_Mixed_Sentiment_Analysis.git repo
%cd /content/repo/ml
!pip install -q peft shap
!pip uninstall -y -q torchao

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
Cloning into 'repo'...
remote: Enumerating objects: 813, done.
remote: Counting objects: 100% (710/710), done.
remote: Compressing objects: 100% (328/328), done.
remote: Total 813 (delta 435), reused 616 (delta 351), pack-reused 103 (from 1)
Receiving objects: 100% (813/813), 91.89 MiB | 24.18 MiB/s, done.
Resolving deltas: 100% (446/446), done.
/content/repo/ml


In [12]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN set:", bool(os.environ["HF_TOKEN"]))

HF_TOKEN set: True


In [13]:
# 2 — data + checkpoint
import os
os.makedirs("data/raw", exist_ok=True)
from google.colab import files
up = files.upload()   # the raw CSV
os.replace("singlish_mixed_sentiment_complete.csv", "data/raw/singlish_mixed_sentiment_complete.csv")
!python -m src.preprocess.quarantine
!python -m src.preprocess.clean_text
!python -m src.preprocess.language_id
!mkdir -p checkpoints
!cp "/content/drive/MyDrive/Final_Reporing_Sentiment_Analysis/thesis_cache_2/checkpoints/lora_scl_lid.pt" checkpoints/


Saving singlish_mixed_sentiment_complete.csv to singlish_mixed_sentiment_complete.csv
raw:        10539
clean:      10509  -> /content/repo/ml/data/interim/clean.csv
quarantine: 30  -> /content/repo/ml/data/interim/quarantine.csv

clean label counts:
sentiment_label
positive    3753
negative    3412
neutral     3344

quarantine reasons:
quarantine_reason
missing_label                               16
non_standard_label:mixed                     6
non_standard_label:label                     4
non_standard_label:sentiment                 1
non_standard_label:neutral-negative          1
non_standard_label:humorous                  1
non_standard_label:sinhala-english mixed     1

--- paste into ml/DATA.md under 'Derived artifacts' ---
## data/interim/clean.csv + data/interim/quarantine.csv
- source:   data/raw/singlish_mixed_sentiment_complete.csv (sha256: 0b513cff6676969b...)
- script:   src/preprocess/quarantine.py @ git commit 2c48912
- output:   clean.csv 10509 rows (sha256: 321216cc

In [14]:
# 3 — run (expect ~30-60 min, mostly SHAP)
!python -m src.explain.run_explain --ckpt checkpoints/lora_scl_lid.pt

device: cuda
Loading weights: 100% 199/199 [00:00<00:00, 6846.80it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
test rows with >=2 words: 1545 of 1577
  SHAP 20/200
  SHAP 40/200
  SHAP 60/200
  SHAP 80/200
  SHAP 100/200
  SHAP 120/200
  SHAP 140/200
  SHAP 160/200
  SHAP 180/200
  SHAP 200/200
SHAP done on 200 samples -> /content/repo/ml/results/explain/shap_values.json
  intensifier harima  n=  6 mean|attr|=0.2271
  intensifier godak   n=  7 mean|attr|=0.1051
  intensifier hari    n=  6 mean|attr|=0.0441
  negation    ne      n= 16

In [15]:
# 4 — bank everything
SAVE = "/content/drive/MyDrive/Final_Reporing_Sentiment_Analysis/thesis_cache_2"
!mkdir -p "{SAVE}/explain"
!cp -r results/explain/* "{SAVE}/explain/"
!cp results/figures/lexicon_attribution.* "{SAVE}/explain/"
from google.colab import files
for x in ["shap_values", "lexicon_attribution", "faithfulness", "minimal_pairs"]:
    files.download(f"results/explain/{x}.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>